In [2]:
import os
import subprocess
import re
import statistics
import itertools

# CUDA code to examined
CUDA_TEMPLATE = r'''

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define BM __BM__
#define BN __BN__
#define BK __BK__
#define TM __TM__
#define TN __TN__
#define NUM_THREADS ((BN / TN) * (BM / TM))

// ---- KERNEL CUDA: 2D Register Tiling ----
__global__ void MatMult(int n, const float *A, const float *B, float *C) {

    // Shared Memory exploited by the Thread Block for the computation of the matrix slice
    __shared__ float sA[BM][BK];
    __shared__ float sB[BK][BN];

    // Local coordinates
    int bx = blockIdx.x;
    int by = blockIdx.y;
    int tx = threadIdx.x;
    int ty = threadIdx.y;

    // Registers to store the partial results of the multiplication, in order to maximize the speed
    float res[TM][TN] = {0.0f};
    float a_reg[TM];
    float b_reg[TN];

    // Linear index of the thread for Decoupled Loading
    int tid = ty * (BN / TN) + tx;

    // Loop: slicing along the k dimension (BK=16)
    for (int p = 0; p < (n + BK - 1) / BK; ++p) {

        // Decoupled Loading, each thread transfers 8 elements ((128x16)/256) from the Global Memory to the Shared Memory Tile A and B in a contiguous way
        // - Linear index idx (Memory Coalescing)
        // - a_row, a_col: 2D coordinates within the Shared Memory
        // - g_a_row, g_a_col: transformation in global coordinates
        // - Saving in Shared Memory sA of the value or padding (0) if out of the matrix borders

        // Loading of a slice of the Tile A
        for (int i = 0; i < (BM * BK) / NUM_THREADS; ++i) {
            int idx = i * NUM_THREADS + tid;
            int a_row = idx / BK;
            int a_col = idx % BK;
            int g_a_row = by * BM + a_row;
            int g_a_col = p * BK + a_col;
            sA[a_row][a_col] = (g_a_row < n && g_a_col < n) ? A[g_a_row * n + g_a_col] : 0.0f;
        }

        // Loading of a slice of the Tile B
        for (int i = 0; i < (BK * BN) / NUM_THREADS; ++i) {
            int idx = i * NUM_THREADS + tid;
            int b_row = idx / BN;
            int b_col = idx % BN;
            int g_b_row = p * BK + b_row;
            int g_b_col = bx * BN + b_col;
            sB[b_row][b_col] = (g_b_row < n && g_b_col < n) ? B[g_b_row * n + g_b_col] : 0.0f;
        }

        // Synchronization Barrier: ensures that all threads have finished loading the data before to start the computation
        __syncthreads();


        // Register Promotion
        // - Moving data from Shared Memory to registers for the computation
        #pragma unroll
        for (int k = 0; k < BK; ++k) {
            for(int i=0; i<TM; i++) a_reg[i] = sA[ty * TM + i][k];
            for(int j=0; j<TN; j++) b_reg[j] = sB[k][tx * TN + j];

            for(int i=0; i<TM; i++) {
                for(int j=0; j<TN; j++) {
                    res[i][j] += a_reg[i] * b_reg[j];
                }
            }
        }
        __syncthreads();
    }


    // Saving in the Global Memory C
    // - Each thread saves its 64 results stored in the Registers to the Global Memory
    int row = by * BM + ty * TM;
    int col = bx * BN + tx * TN;

    for (int i = 0; i < TM; ++i) {
        for (int j = 0; j < TN; ++j) {
            int c_row = row + i;
            int c_col = col + j;
            if (c_row < n && c_col < n) {
                C[c_row * n + c_col] = res[i][j];
            }
        }
    }
}


// ---- HOST ----
int main(int argc, char **argv) {

    if (argc < 2) {
        fprintf(stderr, "Error: missing argument in %s \n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);

    // Host Memory Allocation (1D Flattening)
    size_t bytes = n * n * sizeof(float);
    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    // Initialization
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    // Device Memory Allocation
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Copy from Host to Device
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Execution configuration for the Kernel launch: Grid and Block dimensions
    dim3 threadsPerBlock(BN / TN, BM / TM);
    dim3 blocksPerGrid((n + BN - 1) / BN, (n + BM - 1) / BM);

    printf("Starting the computation...\n");

    // CUDA Event for Timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Kernel Launch + Time Measurement
    cudaEventRecord(start);
    MatMult<<<blocksPerGrid, threadsPerBlock>>>(n, d_a, d_b, d_c);
    cudaEventRecord(stop);

    // Synchronization: Host waits for Device execution
    cudaEventSynchronize(stop);

    // --- Added for the Benchmark ---
    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        printf("CRITICAL_ERROR: %s\n", cudaGetErrorString(err));

        // Free memory and exit before doing anything else
        cudaFree(d_a);
        cudaFree(d_b);
        cudaFree(d_c);
        free(h_a);
        free(h_b);
        free(h_c);
        return 1;
    }


    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Fast Execution Time: %.4f seconds\n", milliseconds / 1000.0f);

    // Copy from Device to Host
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    // Results
    FILE *f = fopen("mat-res-fast.txt", "w");
    if (f) {
        fprintf(f, "%d\n\n", n);
        int limit = (n < 1000) ? n : 1000;
        for (int i = 0; i < limit; i++) {
            for (int j = 0; j < limit; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\n");
        }
        fclose(f);
    }

    // Memory deallocation
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

# Parameters to tune
BLOCK_SIZES = [32, 64, 128]
THREAD_SIZES = [4, 8]
BK_SIZES = [8, 16, 32]

# Settings
MATRIX_SIZE = 15000
NUM_RUNS = 3
CU_FILENAME = "run.cu"
EXE_FILENAME = "./run"

def check_constraints(BM, BN, BK, TM, TN):
    if BM % TM != 0 or BN % TN != 0: return False
    num_threads = (BM // TM) * (BN // TN)
    if num_threads > 1024: return False
    if (BM * BK) % num_threads != 0 or (BK * BN) % num_threads != 0: return False
    shared_mem_bytes = (BM * BK + BK * BN) * 4
    if shared_mem_bytes > 49152: return False
    return True

def run_benchmark():
    combinations = []

    for b_size in BLOCK_SIZES:
        for t_size in THREAD_SIZES:
            for bk in BK_SIZES:
                combinations.append({
                    'BM': b_size,
                    'BN': b_size,
                    'BK': bk,
                    'TM': t_size,
                    'TN': t_size
                })

    results = []

    print(" Starting CUDA Hyperparameter Tuning...")

    for config in combinations:
        if not check_constraints(**config):
            continue

        print(f"\n>>> Test Config: BM={config['BM']}, BN={config['BN']}, BK={config['BK']}, TM={config['TM']}, TN={config['TN']}")

        code = CUDA_TEMPLATE
        for key, val in config.items():
            code = code.replace(f"__{key}__", str(val))

        with open(CU_FILENAME, 'w') as f:
            f.write(code)

        compile_process = subprocess.run(["nvcc", "-arch=sm_75", "-O3", CU_FILENAME, "-o", EXE_FILENAME], capture_output=True)
        if compile_process.returncode != 0:
            print("  [!] Compilation Failed. Skipped.")
            continue

        times = []
        for i in range(1, NUM_RUNS + 1):
            run_process = subprocess.run([EXE_FILENAME, str(MATRIX_SIZE)], capture_output=True, text=True)

            # Check HW Error
            if "CRITICAL_ERROR" in run_process.stdout:
                err_msg = re.search(r"CRITICAL_ERROR:\s+(.*)", run_process.stdout).group(1)
                print(f"  [Run {i}/{NUM_RUNS}] Hardware Error: {err_msg}")
                break

            match = re.search(r"Execution Time:\s+([0-9.]+)\s+seconds", run_process.stdout)
            if match:
                t = float(match.group(1))
                times.append(t)
                print(f"  [Run {i}/{NUM_RUNS}] Time: {t:.4f}s")
            else:
                print(f"  [Run {i}/{NUM_RUNS}] Execution Error Unknown!")

        if len(times) == NUM_RUNS:
            avg_time = statistics.mean(times)
            results.append((config, avg_time))
            print(f"  Average Time: {avg_time:.4f}s")
        else:
            print(f"Configuration Discarded")

    results.sort(key=lambda x: x[1])
    print("\n")
    print(" Results ")
    for i, (config, avg_time) in enumerate(results):
        print(f"{i+1}. Time: {avg_time:.4f}s | Config: {config}")

    if os.path.exists(CU_FILENAME): os.remove(CU_FILENAME)
    if os.path.exists(EXE_FILENAME): os.remove(EXE_FILENAME)

if __name__ == "__main__":
    run_benchmark()

 Starting CUDA Hyperparameter Tuning...

>>> Test Config: BM=32, BN=32, BK=8, TM=4, TN=4
  [Run 1/3] Time: 3.3709s
  [Run 2/3] Time: 3.4688s
  [Run 3/3] Time: 3.4714s
  Average Time: 3.4370s

>>> Test Config: BM=32, BN=32, BK=16, TM=4, TN=4
  [Run 1/3] Time: 3.4369s
  [Run 2/3] Time: 3.4194s
  [Run 3/3] Time: 3.4948s
  Average Time: 3.4504s

>>> Test Config: BM=32, BN=32, BK=32, TM=4, TN=4
  [Run 1/3] Time: 3.4070s
  [Run 2/3] Time: 3.4824s
  [Run 3/3] Time: 3.4764s
  Average Time: 3.4553s

>>> Test Config: BM=32, BN=32, BK=8, TM=8, TN=8
  [Run 1/3] Time: 4.4280s
  [Run 2/3] Time: 4.4130s
  [Run 3/3] Time: 4.4811s
  Average Time: 4.4407s

>>> Test Config: BM=32, BN=32, BK=16, TM=8, TN=8
  [Run 1/3] Time: 4.9650s
  [Run 2/3] Time: 5.0650s
  [Run 3/3] Time: 5.0876s
  Average Time: 5.0392s

>>> Test Config: BM=32, BN=32, BK=32, TM=8, TN=8
  [Run 1/3] Time: 6.1856s
  [Run 2/3] Time: 6.3298s
  [Run 3/3] Time: 6.3201s
  Average Time: 6.2785s

>>> Test Config: BM=64, BN=64, BK=8, TM=4, TN=4
 